# Local-Level Baselines — SARIMAX for Counties (Milton) and Clusters (Helene)

**Prerequisite**: Run `recompute_flows.ipynb` to generate flow time series CSVs.

**Purpose**: Fit SARIMAX baselines on saved flow CSVs and export predictions.
Once baselines are saved, metrics are computed in `local_metrics.ipynb`.

**Outputs**: `results/local_level/{hurricane}/baseline_{flow}_{unit_id}.csv`
Each CSV: date, y_true, y_pred, ci_lower, ci_upper

**Plots**: Predicted vs actual for every local unit (saved to figures/)

In [8]:
import pandas as pd
import numpy as np
import os
import warnings

import matplotlib.pyplot as plt
import matplotlib.dates as mdates

%run ./recovery_function_v2.py

warnings.filterwarnings('ignore')

LOCAL_ROOT = '../results/local_level'

ARIMA_ORDER = (1, 0, 0)
SEASONAL_ORDER = (0, 0, 0, 0)

print('Setup complete.')
print(f'ARIMA: {ARIMA_ORDER}, Seasonal: {SEASONAL_ORDER}')

Setup complete.
ARIMA: (1, 0, 0), Seasonal: (0, 0, 0, 0)


In [9]:
# ── Hurricane configs ──
HURRICANE_CONFIGS = {
    'milton': {
        'landing_date': pd.Timestamp('2024-10-09'),
        'train_end': '2024-08-31',
        'forecast_start': '2024-10-03',
        'forecast_end': '2024-10-31',
        'unit_type': 'county',  # individual counties
    },
    'helene': {
        'landing_date': pd.Timestamp('2024-09-26'),
        'train_end': '2024-09-19',
        'forecast_start': '2024-09-20',
        'forecast_end': '2024-10-31',
        'unit_type': 'cluster',  # spatial clusters
    },
}

FLOW_TYPES = ['within', 'inflow', 'outflow']

In [10]:
def fit_and_export_baselines(hrc_name, hrc_cfg):
    """Fit SARIMAX for all flow types x local units, export CSVs + plots."""
    landing = hrc_cfg['landing_date']
    train_end = hrc_cfg['train_end']
    fc_start = hrc_cfg['forecast_start']
    fc_end = hrc_cfg['forecast_end']
    unit_type = hrc_cfg['unit_type']
    
    hrc_dir = f'{LOCAL_ROOT}/{hrc_name}'
    fig_dir = f'{hrc_dir}/figures'
    os.makedirs(fig_dir, exist_ok=True)
    
    # Load county/cluster names for labeling
    if unit_type == 'county':
        meta = pd.read_csv(f'{hrc_dir}/county_metadata.csv')
        meta['GEOID'] = meta['GEOID'].astype(str)
        id_to_name = dict(zip(meta['GEOID'], meta['NAME']))
    else:
        cluster_df = pd.read_csv(f'{hrc_dir}/county_cluster_assignments.csv')
        cluster_summary = cluster_df.groupby('cluster').agg(
            n_counties=('GEOID', 'count'),
        ).reset_index()
        id_to_name = {str(r['cluster']): f"Cluster {r['cluster']} ({r['n_counties']}co)"
                      for _, r in cluster_summary.iterrows()}
    
    print(f"\n{'='*70}")
    print(f'{hrc_name.upper()} — {len(id_to_name)} {unit_type}s')
    print(f'Landing: {landing.date()}, Train end: {train_end}')
    print(f'Forecast: {fc_start} to {fc_end}')
    print(f"{'='*70}")
    
    summary = []
    
    for ft in FLOW_TYPES:
        # Load flow time series
        flow_path = f'{hrc_dir}/flow_ts_{ft}.csv'
        if not os.path.exists(flow_path):
            print(f'  MISSING: {flow_path}')
            continue
        
        flow_df = pd.read_csv(flow_path, index_col=0, parse_dates=True)
        dates_all = flow_df.index
        unit_ids = flow_df.columns.tolist()
        
        print(f"\n  {'─'*50}")
        print(f'  {ft.upper()}: {len(unit_ids)} units, {len(dates_all)} days')
        print(f"  {'─'*50}")
        
        n_success = 0
        n_fail = 0
        
        for uid in unit_ids:
            flow_y = flow_df[uid].values
            label = id_to_name.get(uid, uid)
            csv_path = f'{hrc_dir}/baseline_{ft}_{uid}.csv'
            fig_path = f'{fig_dir}/baseline_{ft}_{uid}.png'
            
            try:
                y_log, y, X = prepare_time_series_with_exog(flow_y, dates_all)
                res, y_train, X_train = fit_arimax_model(
                    y_log, X,
                    order=ARIMA_ORDER, seasonal_order=SEASONAL_ORDER,
                    train_2024_end=train_end)
                df_rec, forecast_idx = get_predictions_and_ci(
                    res, X, y,
                    forecast_start=fc_start, forecast_end=fc_end)
                
                # Save baseline
                df_rec.to_csv(csv_path)
                
                # Plot predicted vs actual
                fig, ax = plt.subplots(figsize=(14, 5))
                ax.plot(df_rec.index, df_rec['y_true'], 'k-', lw=1.5, label='Observed')
                ax.plot(df_rec.index, df_rec['y_pred'], 'r-', lw=1.5, label='Predicted (baseline)')
                ax.fill_between(df_rec.index, df_rec['ci_lower'], df_rec['ci_upper'],
                               color='red', alpha=0.15, label='95% CI')
                ax.axvline(landing, color='blue', ls='--', lw=2, label='Landing date')
                ax.set_title(f'{hrc_name.capitalize()} — {label} — {ft}\n'
                            f'Predicted vs Actual', fontweight='bold')
                ax.set_xlabel('Date')
                ax.set_ylabel('Mobility (flow)')
                ax.legend(fontsize=9)
                ax.grid(True, alpha=0.3)
                plt.tight_layout()
                plt.savefig(fig_path, dpi=100, bbox_inches='tight')
                plt.close()
                
                n_success += 1
                
                summary.append({
                    'unit_id': uid, 'unit_name': label,
                    'flow_type': ft, 'status': 'success',
                    'pred_mean': df_rec['y_pred'].mean(),
                    'true_mean': df_rec['y_true'].mean(),
                })
                
            except Exception as e:
                n_fail += 1
                print(f'    FAILED {label}: {e}')
                summary.append({
                    'unit_id': uid, 'unit_name': label,
                    'flow_type': ft, 'status': f'failed: {e}',
                })
        
        print(f'    {ft}: {n_success} success, {n_fail} failed')
    
    # Save summary
    pd.DataFrame(summary).to_csv(f'{hrc_dir}/baseline_summary.csv', index=False)
    return summary

## 1. Milton — County-Level Baselines

In [11]:
milton_summary = fit_and_export_baselines('milton', HURRICANE_CONFIGS['milton'])


MILTON — 21 countys
Landing: 2024-10-09, Train end: 2024-08-31
Forecast: 2024-10-03 to 2024-10-31

  ──────────────────────────────────────────────────
  WITHIN: 21 units, 196 days
  ──────────────────────────────────────────────────
    within: 21 success, 0 failed

  ──────────────────────────────────────────────────
  INFLOW: 21 units, 196 days
  ──────────────────────────────────────────────────
    inflow: 21 success, 0 failed

  ──────────────────────────────────────────────────
  OUTFLOW: 21 units, 196 days
  ──────────────────────────────────────────────────
    outflow: 21 success, 0 failed


## 2. Helene — Cluster-Level Baselines

In [12]:
helene_summary = fit_and_export_baselines('helene', HURRICANE_CONFIGS['helene'])


HELENE — 38 clusters
Landing: 2024-09-26, Train end: 2024-09-19
Forecast: 2024-09-20 to 2024-10-31

  ──────────────────────────────────────────────────
  WITHIN: 38 units, 196 days
  ──────────────────────────────────────────────────
    within: 38 success, 0 failed

  ──────────────────────────────────────────────────
  INFLOW: 38 units, 196 days
  ──────────────────────────────────────────────────
    inflow: 38 success, 0 failed

  ──────────────────────────────────────────────────
  OUTFLOW: 38 units, 196 days
  ──────────────────────────────────────────────────
    outflow: 38 success, 0 failed


In [13]:
# ── Verification ──
for hrc_name in HURRICANE_CONFIGS:
    hrc_dir = f'{LOCAL_ROOT}/{hrc_name}'
    files = sorted([f for f in os.listdir(hrc_dir) if f.startswith('baseline_') and f.endswith('.csv')])
    print(f'\n{hrc_name}: {len(files)} baseline CSVs')
    for ft in FLOW_TYPES:
        ft_files = [f for f in files if f.startswith(f'baseline_{ft}_')]
        print(f'  {ft}: {len(ft_files)}')


milton: 64 baseline CSVs
  within: 21
  inflow: 21
  outflow: 21

helene: 115 baseline CSVs
  within: 38
  inflow: 38
  outflow: 38
